# RFID-ExSim 데이터셋 분석 & 시각화

> 고등학생 데이터 기반 탐구 프로젝트용 노트북
> 데이터 출처: [RFID-ExSim-Dataset v1.0](https://github.com/yasserhmimou9/RFID-ExSim-Dataset/tree/v1.0) (MIT License, Yasser Hmimou)

## 이 데이터는 무엇인가요?
교통카드·출입카드처럼 **RFID 태그**를 리더기에 대면 "읽힘(read)" 기록이 남습니다.
이 데이터셋은 ESP32 + MFRC522 리더기 2대(`ESP32_A`, `ESP32_B`)와 태그 12개(`TAG01`~`TAG12`)로
**정상 상황과 공격 상황**을 실험해서 40만 건 이상의 읽기 기록을 모은 것입니다.

| 시나리오 | 의미 | 탐구 포인트 |
|---|---|---|
| **S1** 정상 읽기 | 거리 1–4 cm, 각도 0°/45°/90° | 거리·각도가 인식에 미치는 영향 |
| **S2** 동시 읽기(충돌) | 리더 A·B가 같은 태그를 동시에 읽음 | 두 리더의 시간 차이, 충돌 |
| **S3** 태그 복제 | 서로 다른 태그 2개가 같은 UID를 가짐 | 복제 카드를 데이터로 찾아낼 수 있나? |
| **S4** 재전송(Replay) 공격 | 녹화한 신호를 다시 보내거나 가짜 기록 주입 | 정상과 다른 시간 패턴 |
| **S5** 폭주(DoS 유사) | 매우 빠른 연속 읽기 | 초당 읽기 수 폭증 |

## 사용 방법
1. README의 OneDrive 링크에서 데이터를 내려받아 압축을 풉니다.
2. 아래 `DATA_DIR`에 압축을 푼 폴더 경로를 적습니다. (Google Colab이면 파일을 업로드하거나 Drive를 연결)
3. 위에서부터 차례로 셀을 실행합니다. (`Shift + Enter`)

## 0. 준비: 라이브러리와 한글 폰트  `→ 보고서 부록 B`

In [ ]:
# Colab/로컬 공통: 그래프에 한글이 보이도록 폰트 패키지 설치 (처음 한 번만)
%pip install -q koreanize-matplotlib

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import koreanize_matplotlib  # noqa: F401  (import만 하면 한글 폰트가 적용됨)
except ImportError:
    print("한글 폰트 패키지가 없어 글자가 네모로 보일 수 있어요.")

plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.unicode_minus": False,
})

# 시나리오/리더기마다 항상 같은 색을 쓰도록 고정 (색약자도 구분 가능한 팔레트)
SCENARIO_COLORS = {"S1": "#2a78d6", "S2": "#eb6834", "S3": "#1baf7a", "S4": "#eda100", "S5": "#e87ba4"}
READER_COLORS = {"ESP32_A": "#2a78d6", "ESP32_B": "#eb6834"}
SCENARIO_NAMES = {"S1": "S1 정상", "S2": "S2 충돌", "S3": "S3 복제", "S4": "S4 재전송", "S5": "S5 폭주"}

## 1. 데이터 불러오기  `→ 보고서 Ⅱ-1`
`data/processed/rfid_dataset_S1_all.jsonl` ~ `S5` 파일을 찾아 하나의 표(DataFrame)로 합칩니다.
JSONL은 **한 줄에 기록 1개(JSON)** 가 들어 있는 형식이에요.

In [ ]:
# ▼ 압축을 푼 폴더 경로로 바꾸세요. (예: Colab 업로드 시 "/content/rfid_dataset")
DATA_DIR = Path("rfid_dataset")

files = sorted(DATA_DIR.rglob("rfid_dataset_S*_all.jsonl"))
assert files, f"{DATA_DIR.resolve()} 아래에서 rfid_dataset_S*_all.jsonl 파일을 찾지 못했어요. 경로를 확인하세요."
for f in files:
    print(f"{f.name:30s} {f.stat().st_size / 1e6:7.1f} MB")

In [ ]:
frames = []
for f in files:
    part = pd.read_json(f, lines=True, dtype=False)
    part["source_file"] = f.name
    frames.append(part)
df = pd.concat(frames, ignore_index=True)

# 시간 문자열 → 날짜/시간 형식, 숫자 열 → 숫자 형식
df["timestamp"] = pd.to_datetime(df["timestamp_iso"], format="ISO8601", utc=True, errors="coerce")
for col in ["distance_cm", "orientation_deg"]:
    if col in df:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# session_id가 비어 있으면 (시나리오, 태그, 파일)로 대신 구분
if "session_id" not in df or df["session_id"].isna().all():
    df["session_id"] = df["scenario"].astype(str) + "_" + df["tag_local_id"].astype(str)
df["session_id"] = df["session_id"].fillna(df["scenario"].astype(str) + "_" + df["tag_local_id"].astype(str))

df = df.sort_values(["session_id", "timestamp"]).reset_index(drop=True)
print("전체 기록 수:", f"{len(df):,}")
df.head()

## 2. 데이터 둘러보기 (EDA)  `→ 보고서 Ⅱ-1, Ⅱ-2, Ⅲ-1`
분석 전에 **어떤 열이 있고, 빈 값은 없는지, 값의 종류는 무엇인지** 먼저 확인합니다.

In [ ]:
summary = pd.DataFrame({
    "자료형": df.dtypes.astype(str),
    "빈 값 수": df.isna().sum(),
    "고유값 수": df.nunique(dropna=True),
})
summary

In [ ]:
# 범주형 열의 값 종류 확인 (event 값은 보고서에서 꼭 설명하세요)
for col in ["scenario", "device_id", "tag_local_id", "event", "distance_cm", "orientation_deg"]:
    if col in df:
        print(f"── {col}")
        print(df[col].value_counts(dropna=False).head(15).to_string(), "\n")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

counts = df["scenario"].value_counts().sort_index()
axes[0].bar(counts.index, counts.values, color=[SCENARIO_COLORS.get(s, "gray") for s in counts.index])
axes[0].set_title("시나리오별 기록 수")
for x, v in zip(counts.index, counts.values):
    axes[0].annotate(f"{v:,}", (x, v), ha="center", va="bottom", fontsize=9)

dev = df["device_id"].value_counts().sort_index()
axes[1].bar(dev.index, dev.values, color=[READER_COLORS.get(d, "gray") for d in dev.index])
axes[1].set_title("리더기별 기록 수")

tag = df["tag_local_id"].value_counts().sort_index()
axes[2].bar(tag.index, tag.values, color="#2a78d6")
axes[2].set_title("태그별 기록 수")
axes[2].tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

### 핵심 파생 변수 만들기  `→ 보고서 Ⅱ-2 (변환), Ⅱ-3`
- **읽기 간격(interval)**: 같은 세션·같은 리더기에서 *직전 읽기와 지금 읽기 사이의 시간(ms)*.
  정상이면 일정하고, 공격이면 너무 짧거나 불규칙할 수 있어요.
- **세션 요약표**: 세션 하나(한 번의 실험)마다 기록 수, 지속 시간, 초당 읽기 수 등을 계산합니다.

In [ ]:
df["interval_ms"] = (
    df.groupby(["session_id", "device_id"])["timestamp"].diff().dt.total_seconds() * 1000
)

def summarize_session(g):
    dur = (g["timestamp"].max() - g["timestamp"].min()).total_seconds()
    iv = g["interval_ms"].dropna()
    return pd.Series({
        "scenario": g["scenario"].iloc[0],
        "tag": g["tag_local_id"].iloc[0],
        "distance_cm": g["distance_cm"].iloc[0] if "distance_cm" in g else np.nan,
        "orientation_deg": g["orientation_deg"].iloc[0] if "orientation_deg" in g else np.nan,
        "n_reads": len(g),
        "duration_s": dur,
        "reads_per_s": len(g) / dur if dur > 0 else np.nan,
        "n_readers": g["device_id"].nunique(),
        "n_tags": g["tag_local_id"].nunique(),
        "n_uids": g["uid_hash"].nunique(),
        "iv_median_ms": iv.median(),
        "iv_mean_ms": iv.mean(),
        "iv_std_ms": iv.std(),
        "iv_cv": iv.std() / iv.mean() if iv.mean() > 0 else np.nan,  # 변동계수: 간격이 얼마나 불규칙한가
        "share_reader_A": (g["device_id"] == "ESP32_A").mean(),
    })

sessions = df.groupby("session_id").apply(summarize_session, include_groups=False).reset_index()
print("세션 수:", len(sessions))
sessions.groupby("scenario")[["n_reads", "duration_s", "reads_per_s", "iv_median_ms"]].median().round(2)

## 3. [S1] 거리와 각도는 인식 속도에 영향을 줄까?  `→ 보고서 Ⅲ-2, Ⅲ-3`
**가설 예시**: "태그가 멀어질수록, 그리고 비스듬할수록 초당 읽기 수가 줄어들 것이다."

In [ ]:
s1 = sessions[sessions["scenario"] == "S1"].dropna(subset=["distance_cm"])
if s1.empty:
    print("S1 세션에 거리 정보가 없어요.")
else:
    pivot = s1.pivot_table(index="orientation_deg", columns="distance_cm", values="reads_per_s", aggfunc="mean")
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    # (1) 거리별 평균 초당 읽기 수 (각도별 선)
    ORIENT_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
    for i, (deg, row) in enumerate(pivot.iterrows()):
        axes[0].plot(row.index, row.values, marker="o", lw=2, color=ORIENT_COLORS[i % 4], label=f"{deg:g}°")
    axes[0].set_xlabel("거리 (cm)")
    axes[0].set_ylabel("평균 초당 읽기 수")
    axes[0].set_title("S1: 거리·각도에 따른 읽기 속도")
    axes[0].legend(title="각도")

    # (2) 히트맵: 진할수록 잘 읽힘
    im = axes[1].imshow(pivot.values, cmap="Blues", aspect="auto")
    axes[1].set_xticks(range(len(pivot.columns)), [f"{c:g}cm" for c in pivot.columns])
    axes[1].set_yticks(range(len(pivot.index)), [f"{r:g}°" for r in pivot.index])
    for (r, c), v in np.ndenumerate(pivot.values):
        if not np.isnan(v):
            axes[1].text(c, r, f"{v:.1f}", ha="center", va="center",
                         color="white" if v > np.nanmean(pivot.values) else "black")
    axes[1].set_title("S1: 초당 읽기 수 히트맵")
    axes[1].grid(False)
    fig.colorbar(im, ax=axes[1], label="초당 읽기 수")
    plt.tight_layout()
    plt.show()
    display(pivot.round(2))

In [ ]:
# 거리별 읽기 간격 분포 (상자그림) — 가운데 선이 중앙값
s1_reads = df[(df["scenario"] == "S1") & df["distance_cm"].notna() & df["interval_ms"].notna()]
if not s1_reads.empty:
    dists = sorted(s1_reads["distance_cm"].unique())
    data = [s1_reads.loc[s1_reads["distance_cm"] == d, "interval_ms"] for d in dists]
    plt.boxplot(data, tick_labels=[f"{d:g}cm" for d in dists], showfliers=False)
    plt.ylabel("읽기 간격 (ms)")
    plt.title("S1: 거리별 읽기 간격 (이상치 제외)")
    plt.show()

    # 상관계수: -1에 가까우면 '거리↑ → 속도↓'
    r = s1[["distance_cm", "reads_per_s"]].corr().iloc[0, 1]
    print(f"거리와 초당 읽기 수의 상관계수 r = {r:.3f}")

## 4. [S2] 두 리더기가 동시에 읽으면 어떻게 될까?  `→ 보고서 Ⅲ-2, Ⅲ-4`
각 A 리더 읽기마다 **가장 가까운 B 리더 읽기까지의 시간차**를 구합니다.
시간차가 아주 작으면(예: 50ms 이내) 두 리더가 "거의 동시에" 읽은 것 = 충돌 후보.

In [ ]:
s2 = df[df["scenario"] == "S2"].dropna(subset=["timestamp"])
gaps = []
for sid, g in s2.groupby("session_id"):
    a = g[g["device_id"] == "ESP32_A"][["timestamp"]].sort_values("timestamp")
    b = g[g["device_id"] == "ESP32_B"][["timestamp"]].sort_values("timestamp").rename(columns={"timestamp": "ts_b"})
    if a.empty or b.empty:
        continue
    m = pd.merge_asof(a, b, left_on="timestamp", right_on="ts_b", direction="nearest")
    gaps.append(((m["ts_b"] - m["timestamp"]).dt.total_seconds() * 1000).abs())

if gaps:
    gap = pd.concat(gaps)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(gap.clip(upper=gap.quantile(0.99)), bins=60, color="#eb6834", edgecolor="white")
    axes[0].set_xlabel("A–B 최근접 읽기 시간차 (ms)")
    axes[0].set_ylabel("빈도")
    axes[0].set_title("S2: 두 리더기의 읽기 시간차 분포")

    share = sessions[sessions["scenario"] == "S2"]["share_reader_A"]
    axes[1].hist(share, bins=20, range=(0, 1), color="#2a78d6", edgecolor="white")
    axes[1].axvline(0.5, color="black", ls="--", lw=1)
    axes[1].set_xlabel("세션 안에서 리더 A가 차지한 비율")
    axes[1].set_title("S2: 어느 리더가 더 많이 읽었나? (0.5 = 공평)")
    plt.tight_layout()
    plt.show()
    for t in [10, 50, 100]:
        print(f"시간차 {t:>3}ms 이내 비율: {(gap <= t).mean():.1%}")
else:
    print("S2에서 A·B 두 리더 기록이 모두 있는 세션이 없어요.")

## 5. [S3] 복제 태그를 데이터로 찾아낼 수 있을까?  `→ 보고서 Ⅲ-4`
정상이라면 **UID 하나 = 실물 태그 하나**입니다.
같은 `uid_hash`가 서로 다른 `tag_local_id`(실물 태그)에서 나타나면 복제가 의심됩니다.

In [ ]:
uid_map = (
    df.groupby("uid_hash")
      .agg(실물태그수=("tag_local_id", "nunique"),
           태그목록=("tag_local_id", lambda s: ", ".join(sorted(s.dropna().unique()))),
           시나리오=("scenario", lambda s: ", ".join(sorted(s.dropna().unique()))),
           기록수=("uid_hash", "size"))
      .sort_values("실물태그수", ascending=False)
)
suspects = uid_map[uid_map["실물태그수"] > 1]
print(f"UID 종류: {len(uid_map)}개, 2개 이상의 실물 태그에서 나온 UID(복제 의심): {len(suspects)}개")
display(suspects.head(10))

In [ ]:
s3 = df[df["scenario"] == "S3"]
if not s3.empty:
    tab = pd.crosstab(s3["tag_local_id"], s3["device_id"])
    ax = tab.plot(kind="bar", color=[READER_COLORS.get(c, "gray") for c in tab.columns],
                  edgecolor="white", figsize=(10, 4))
    ax.set_title("S3: 태그·리더기별 읽기 수")
    ax.set_xlabel("실물 태그")
    ax.set_ylabel("읽기 수")
    ax.legend(title="리더기")
    plt.show()

## 6. [S4] 재전송(Replay) 공격은 시간 패턴이 다를까?  `→ 보고서 Ⅲ-2`
사람이 카드를 대는 정상 상황(S1)과 비교해서, S4의 **읽기 간격 분포**가 어떻게 다른지 봅니다.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
bins = np.logspace(0, 5, 60)  # 1ms ~ 100초를 로그 눈금으로
for sc in ["S1", "S4"]:
    iv = df.loc[(df["scenario"] == sc) & (df["interval_ms"] > 0), "interval_ms"]
    if not iv.empty:
        ax.hist(iv, bins=bins, density=True, histtype="step", lw=2,
                color=SCENARIO_COLORS[sc], label=SCENARIO_NAMES[sc])
ax.set_xscale("log")
ax.set_xlabel("읽기 간격 (ms, 로그 눈금)")
ax.set_ylabel("비율 밀도")
ax.set_title("정상(S1) vs 재전송(S4) 읽기 간격 분포")
ax.legend()
plt.show()

if "event" in df:
    display(pd.crosstab(df["scenario"], df["event"]))

## 7. [S5] 폭주(Flooding) 공격: 초당 읽기 수가 얼마나 치솟나?  `→ 보고서 Ⅲ-3 (시간 변화)`
가장 기록이 많은 S5 세션 하나를 골라 **1초 단위로 읽기 수**를 세어 시계열로 그립니다.

In [ ]:
s5_sessions = sessions[sessions["scenario"] == "S5"].sort_values("n_reads", ascending=False)
if not s5_sessions.empty:
    sid = s5_sessions["session_id"].iloc[0]
    g = df[df["session_id"] == sid].set_index("timestamp")
    fig, ax = plt.subplots(figsize=(12, 4))
    for dev, gg in g.groupby("device_id"):
        per_sec = gg.resample("1s").size()
        t = (per_sec.index - g.index.min()).total_seconds()
        ax.plot(t, per_sec.values, lw=2, color=READER_COLORS.get(dev, "gray"), label=dev)
    # 비교 기준선: S1 세션들의 평균 초당 읽기 수
    base = sessions.loc[sessions["scenario"] == "S1", "reads_per_s"].mean()
    if not np.isnan(base):
        ax.axhline(base, color="black", ls="--", lw=1)
        ax.annotate(f"S1 평균 {base:.1f}회/초", (0, base), xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel("세션 시작 후 경과 시간 (초)")
    ax.set_ylabel("초당 읽기 수")
    ax.set_title(f"S5 폭주 세션 시계열: {sid}")
    ax.legend(title="리더기")
    plt.show()

## 8. 다섯 시나리오 한눈에 비교  `→ 보고서 Ⅲ-2, Ⅳ-1`
세션마다 계산한 지표를 시나리오별 상자그림으로 비교합니다. **어떤 지표가 공격을 가장 잘 구별할까요?**

In [ ]:
metrics = {"reads_per_s": "초당 읽기 수", "iv_median_ms": "읽기 간격 중앙값 (ms)",
           "iv_cv": "간격 변동계수 (불규칙성)", "n_reads": "세션당 기록 수"}
order = [s for s in SCENARIO_COLORS if s in sessions["scenario"].unique()]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (col, label) in zip(axes.flat, metrics.items()):
    data = [sessions.loc[sessions["scenario"] == s, col].dropna() for s in order]
    bp = ax.boxplot(data, tick_labels=[SCENARIO_NAMES[s] for s in order], patch_artist=True, showfliers=False,
                    medianprops={"color": "black"})
    for patch, s in zip(bp["boxes"], order):
        patch.set_facecolor(SCENARIO_COLORS[s])
        patch.set_alpha(0.7)
    ax.set_title(label)
    if col in ("reads_per_s", "iv_median_ms", "n_reads"):
        ax.set_yscale("log")
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
plt.tight_layout()
plt.show()
sessions.groupby("scenario")[list(metrics)].median().round(3)

## 9. (심화) 머신러닝으로 시나리오 맞히기  `→ 보고서 Ⅲ-3 (모델), Ⅴ-1`
세션 요약 지표만 보고 **어떤 시나리오(정상/공격 종류)인지 맞히는 모델**을 만들어 봅니다.
결정 트리(Decision Tree)는 "초당 읽기 수가 ○○보다 크면 → S5"처럼 규칙을 직접 보여줘서 이해하기 쉬워요.

> 주의: 학습에 쓴 세션과 평가에 쓴 세션을 반드시 분리해야 정직한 정확도가 나옵니다.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

features = ["reads_per_s", "iv_median_ms", "iv_mean_ms", "iv_std_ms", "iv_cv",
            "n_reads", "duration_s", "n_readers", "n_tags", "n_uids", "share_reader_A"]
data = sessions.dropna(subset=["scenario"]).copy()
X = data[features].fillna(0)
y = data["scenario"]

strat = y if y.value_counts().min() >= 2 else None
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=strat)
model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)
pred = model.predict(X_te)
print(f"학습 세션 {len(X_tr)}개, 평가 세션 {len(X_te)}개")
print(f"정확도: {accuracy_score(y_te, pred):.1%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(y_te, pred, ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("혼동 행렬 (행: 실제, 열: 예측)")
axes[0].grid(False)
imp = pd.Series(model.feature_importances_, index=features).sort_values()
axes[1].barh(imp.index, imp.values, color="#2a78d6")
axes[1].set_title("어떤 지표가 판단에 중요했나? (특성 중요도)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 7))
plot_tree(model, feature_names=features, class_names=model.classes_, filled=True, rounded=True, fontsize=9)
plt.title("결정 트리가 찾아낸 규칙")
plt.show()

## 10. 결과 저장  `→ 보고서 부록 A`
보고서에 넣을 표를 CSV로 저장합니다. (그래프는 각 그림을 마우스 오른쪽 클릭 → 이미지 저장, 또는 `plt.savefig("파일명.png", dpi=200)`)

In [ ]:
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
sessions.to_csv(OUT / "session_summary.csv", index=False, encoding="utf-8-sig")  # 엑셀에서 한글이 안 깨지도록
print("저장 완료:", (OUT / "session_summary.csv").resolve())

## 다음 단계: 보고서 쓰기
각 절 제목 옆의 `→ 보고서 Ⅲ-2` 표시는 결과를 넣을 **보고서 양식의 칸**입니다.
절별로 무엇을 쓸지는 `보고서_작성_가이드.md`에 정리되어 있어요.
그래프를 붙일 때는 **그래프에서 읽은 사실을 숫자와 함께 문장으로** 함께 쓰세요.